In [1]:
import os
import uuid
import re
from typing import Any

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore


# ================= 1. 模型 =================
model = ChatOpenAI(
    model="qwen-plus",
    openai_api_key=os.getenv("DASHSCOPE_API_KEY"),
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# ================= ❗关键：不用 embedding =================
store = InMemoryStore()   # ✅ 纯KV


# ================= 2. 输入清洗 =================
def safe_get_text(content: Any) -> str:
    if content is None:
        return ""

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        texts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                texts.append(item.get("text", ""))
            elif isinstance(item, str):
                texts.append(item)
        return "".join(texts).strip()

    if isinstance(content, dict):
        return content.get("text", "")

    return str(content).strip()


# ================= 3. 信息抽取 =================
def extract_name(text: str):
    if not text:
        return None

    patterns = [
        r"我叫([a-zA-Z0-9_]+)",
        r"我的名字叫([a-zA-Z0-9_]+)",
        r"my name is ([a-zA-Z0-9_]+)",
        r"i am ([a-zA-Z0-9_]+)"
    ]

    for p in patterns:
        m = re.search(p, text.lower())
        if m:
            return m.group(1)

    return None


# ================= 4. KV记忆 =================

def save_memory(store: BaseStore, namespace, key, value):
    if not value:
        return

    # 👉 直接覆盖（不用 search）
    store.put(namespace, key, {"type": key, "value": value})
    print(f"[记忆写入] {key}={value}")


def load_memory(store: BaseStore, namespace):
    result = {}

    try:
        # 👉 直接按 key 读取（最稳定）
        item = store.get(namespace, "name")

        if item and isinstance(item.value, dict):
            result["name"] = item.value.get("value")

    except Exception as e:
        print("[load memory error]", e)

    return result


# ================= 5. 核心节点 =================
def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):

    user_id = config.get("configurable", {}).get("user_id", "default")
    namespace = ("memories", user_id)

    # ===== 输入 =====
    last_msg = state["messages"][-1]
    query_text = safe_get_text(last_msg.content)

    # ===== 自动记忆 =====
    name = extract_name(query_text)
    if name:
        save_memory(store, namespace, "name", name)

    # ===== 读取记忆 =====
    memory = load_memory(store, namespace)
    memory_text = "\n".join([f"{k}: {v}" for k, v in memory.items()])

    # ===== Prompt =====
    messages = [
        SystemMessage(content="你是一个有长期记忆的AI助手"),
        SystemMessage(content=f"用户信息：\n{memory_text or '暂无'}"),
    ] + state["messages"]

    # ===== 调用模型 =====
    response = model.invoke(messages)

    return {"messages": response}


# ================= 6. Graph =================
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

graph = builder.compile(
    checkpointer=MemorySaver(),
    store=store
)


# ================= 7. 测试 =================
def run_test(thread_id, user_id, text):
    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    input_msg = {"role": "user", "content": text}

    result = None
    for chunk in graph.stream({"messages": [input_msg]}, config, stream_mode="values"):
        result = chunk

    if result:
        result["messages"][-1].pretty_print()


print("\n===== 测试1：写入名字 =====")
run_test("t1", "u1", "我叫tomiezhang")

print("\n===== 测试2：跨线程读取 =====")
run_test("t2", "u1", "我的名字是什么？")


===== 测试1：写入名字 =====
[记忆写入] name=tomiezhang
================================== Ai Message ==================================

你好，tomiezhang！很高兴认识你～😊  
如果你有任何问题、需要帮助（比如学习、工作、编程、写作、生活建议等），或者只是想聊聊天，我都很乐意陪你一起探讨。  
需要我帮你记下什么特别的偏好、习惯或常用信息吗？我可以长期记住，方便以后更贴心地协助你 🌟

===== 测试2：跨线程读取 =====
================================== Ai Message ==================================

你的名字是 **tomiezhang**。
